# Cache invalidation
Update the source of truth, then remove stale cached data.


In [ ]:
database = {1: "Ada"}
cache = {1: "Ada"}

def rename(user_id: int, name: str) -> None:
    database[user_id] = name
    cache.pop(user_id, None)

rename(1, "Grace")
print(cache, database)


## Polished version
Put the write-and-invalidate rule in one application service.


In [ ]:
from typing import Protocol

class UserRepository(Protocol):
    async def rename(self, user_id: int, name: str) -> None: ...

class Cache(Protocol):
    async def delete(self, key: str) -> None: ...

class MemoryUsers:
    def __init__(self) -> None:
        self.users = {1: "Ada"}
    async def rename(self, user_id: int, name: str) -> None:
        self.users[user_id] = name

class MemoryCache:
    def __init__(self) -> None:
        self.values = {"user:1": "Ada"}
    async def delete(self, key: str) -> None:
        self.values.pop(key, None)

class RenameUserService:
    def __init__(self, users: UserRepository, cache_store: Cache) -> None:
        self.users = users
        self.cache = cache_store
    async def rename(self, user_id: int, name: str) -> None:
        await self.users.rename(user_id, name)
        await self.cache.delete(f"user:{user_id}")

users, cache_store = MemoryUsers(), MemoryCache()
await RenameUserService(users, cache_store).rename(1, "Grace")
print(users.users, cache_store.values)
